# 🍎 เทรน G1 บน MacBook (CPU) — ได้โมเดล .pt ไปใช้งาน

notebook นี้เทรน policy ให้ G1 เดิน **บนเครื่อง Mac นี้เลย** ไม่ต้องใช้ Colab
ไม่ต้องใช้ GPU. ช้ากว่า GPU แต่รันงานเล็กๆ ได้โมเดลจริงออกมาใช้

**ต่างจาก `train_g1_colab.ipynb` ยังไง:**
- ไม่ต้อง clone/pip — ใช้ mjlab ที่ติดตั้งใน `.venv` ของโปรเจคอยู่แล้ว
- บังคับโหมด CPU (`CUDA_VISIBLE_DEVICES=''`)
- เทรนรอบน้อยลง + env น้อยลง ให้เหมาะกับ CPU

**⚠️ ก่อนรัน:** ที่มุมขวาบนของ VS Code เลือก kernel เป็น **`.venv`** ของ
โปรเจค mjlab (Python จาก `.venv/bin/python`) ไม่งั้นจะ import mjlab ไม่เจอ

**เวลาโดยประมาณ:** ~3-4 วินาที/รอบ → 300 รอบ ≈ **15-20 นาที**

## 1) บังคับโหมด CPU + ตั้งค่าพื้นฐาน

**สำคัญ:** train script ของ mjlab ถูกออกแบบมาสำหรับ GPU. เราตั้ง
`CUDA_VISIBLE_DEVICES=''` (ว่าง) ซึ่งโค้ดตีความว่า 'ใช้ CPU' — ต้องตั้ง
**ก่อน** import อะไรที่แตะ CUDA

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''  # บังคับ CPU mode

from pathlib import Path
import subprocess, sys

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
print('repo root:', REPO)
print('python   :', sys.executable)

## 2) เทรน! (หัวใจ)

เรียก train script เป็น subprocess (สะดวกกว่ารันในเคอร์เนลตรงๆ และเห็น log
สดๆ). พารามิเตอร์:
- `Mjlab-Velocity-Flat-Unitree-G1` — G1 เดินตามคำสั่งความเร็ว
- `--env.scene.num-envs 256` — 256 ตัวขนาน (CPU ไหว; GPU ใช้เป็นพัน)
- `--agent.max-iterations 300` — 300 รอบ (~15-20 นาที)
- `--agent.save-interval 50` — เซฟทุก 50 รอบ
- `--agent.logger tensorboard` — เลี่ยง wandb (ไม่ต้อง login)

**ดู `Mean reward` ในแต่ละรอบ — ควรค่อยๆ เพิ่มขึ้น** = policy กำลังเรียนรู้

> รอบแรกจะช้าเป็นพิเศษเพราะ MuJoCo Warp ต้อง compile kernel — เป็นเรื่องปกติ

In [ ]:
cmd = [
    sys.executable, '-m', 'mjlab.scripts.train',
    'Mjlab-Velocity-Flat-Unitree-G1',
    '--env.scene.num-envs', '256',
    '--agent.max-iterations', '300',
    '--agent.save-interval', '50',
    '--agent.logger', 'tensorboard',
]
print('รัน:', ' '.join(cmd[2:]))
print('=' * 60)

# สตรีม log สดๆ ระหว่างเทรน
proc = subprocess.Popen(
    cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env={**os.environ, 'CUDA_VISIBLE_DEVICES': ''},
)
for line in proc.stdout:
    # แสดงเฉพาะบรรทัดสำคัญ ไม่ให้ล้นจอ
    if any(k in line for k in ('Mean reward', 'Iteration', 'Learning iteration',
                               'ETA', 'Storing', 'error', 'Error', 'Traceback')):
        print(line.rstrip())
proc.wait()
print('=' * 60)
print('เทรนจบ, exit code =', proc.returncode)

## 3) หาไฟล์โมเดล .pt ที่เทรนได้

In [ ]:
log_dir = REPO / 'logs' / 'rsl_rl' / 'g1_velocity'
runs = sorted(log_dir.glob('*'), key=os.path.getmtime, reverse=True)
assert runs, 'ไม่พบ run — เทรนสำเร็จหรือยัง?'
latest = runs[0]
ckpts = sorted(latest.glob('model_*.pt'),
               key=lambda p: int(''.join(filter(str.isdigit, p.stem))))
checkpoint = str(ckpts[-1])
print('run ล่าสุด :', latest.name)
print('checkpoints:', [c.name for c in ckpts])
print()
print('✅ โมเดลที่ใช้งานได้:', checkpoint)
print('   ขนาด:', round(Path(checkpoint).stat().st_size / 1e6, 1), 'MB')

## 4) ทดสอบโมเดล — เรนเดอร์วิดีโอให้ policy สั่งหุ่นเดิน

โหลด checkpoint กลับมาเป็น policy แล้วให้มันสั่งหุ่นจริง (ไม่ใช่ random)
เรนเดอร์เป็นวิดีโอดูว่าเดินได้แค่ไหน

> เทรนแค่ 300 รอบบน CPU หุ่นอาจยังเดินไม่สวย (โซเซ/ยังไม่มั่นคง) — ปกติ
> เพราะงานจริงใช้หลายพันรอบบน GPU. จุดสำคัญคือ **pipeline ทำงานครบ**

In [ ]:
import torch, numpy as np, mujoco, imageio
from dataclasses import asdict
import mjlab.tasks  # noqa: F401
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg, load_runner_cls
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import RslRlVecEnvWrapper
from mjlab.rl.runner import MjlabOnPolicyRunner

TASK = 'Mjlab-Velocity-Flat-Unitree-G1'
device = 'cpu'

env_cfg = load_env_cfg(TASK, play=True)
env_cfg.scene.num_envs = 1
eval_env = ManagerBasedRlEnv(cfg=env_cfg, device=device, render_mode='rgb_array')

agent_cfg = load_rl_cfg(TASK)
runner_cls = load_runner_cls(TASK) or MjlabOnPolicyRunner
wrapped = RslRlVecEnvWrapper(eval_env, clip_actions=agent_cfg.clip_actions)
runner = runner_cls(wrapped, asdict(agent_cfg), device=device)
runner.load(checkpoint, load_cfg={'actor': True}, strict=True, map_location=device)
policy = runner.get_inference_policy(device=device)
print('✓ โหลด policy จาก', Path(checkpoint).name)

In [ ]:
obs = wrapped.get_observations()
frames = []
for step in range(150):
    with torch.inference_mode():
        action = policy(obs)
    obs, _, _, _ = wrapped.step(action)
    frames.append(eval_env.render())

out = str(REPO / 'g1_trained_mac.mp4')
imageio.mimsave(out, frames, fps=30)
print(f'✓ อัดวิดีโอ {len(frames)} เฟรม -> {out}')

In [ ]:
from IPython.display import Video
Video(out, embed=True, width=480)

## 5) โมเดลอยู่ที่ไหน + เอาไปใช้ยังไง

ไฟล์ `.pt` อยู่ที่ path ที่ปรินต์ในขั้นที่ 3 (ใน `logs/rsl_rl/g1_velocity/...`)
**นี่คือโมเดลที่ใช้งานได้จริง** — ก็อปไปใช้ที่ไหนก็ได้

**เล่นดูแบบ interactive** (viser viewer บนเบราว์เซอร์):
```bash
CUDA_VISIBLE_DEVICES='' uv run play Mjlab-Velocity-Flat-Unitree-G1 \
    --checkpoint-file <path ที่ปรินต์ในขั้น 3> --viewer viser --num-envs 1
```

**อยากให้เดินสวยขึ้น?** เพิ่ม `--agent.max-iterations` (เช่น 1000+) แล้วเทรนใหม่
— แต่บน CPU จะนานขึ้นตามสัดส่วน (1000 รอบ ≈ ชั่วโมง). ถ้าอยากได้ผลระดับ
งานจริง (หลายพันรอบ เร็ว) แนะนำเทรนบนเครื่องที่มี NVIDIA GPU

**อยากเทรนงานหยิบของ?** เปลี่ยน `TASK`/task id เป็น `Mjlab-Lift-Cube-G1`
และแก้ path เป็น `g1_lift_cube` ในขั้นที่ 3